In [3]:

%pip install -q osmnx==2.1.1

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import rasterio
import osmnx as ox
import kagglehub

from pathlib import Path

INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/geo_data")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ox.settings.use_cache = True
ox.settings.requests_timeout = 240

print("Environment initialized.")
print("OSMnx version:", ox.__version__)
print("GeoPandas version:", gpd.__version__)
print("Rasterio version:", rasterio.__version__)

Environment initialized.
OSMnx version: 2.1.1
GeoPandas version: 1.2.0
Rasterio version: 1.5.2


In [ ]:
from pathlib import Path

BASE_DIR = Path("/kaggle/working/geo_data")

DIRS = {
    "boundaries": BASE_DIR / "boundaries",
    "schools_raw": BASE_DIR / "schools_raw",
    "schools_processed": BASE_DIR / "schools_processed",
    "population": BASE_DIR / "population",
    "outputs": BASE_DIR / "outputs",
    "logs": BASE_DIR / "logs"
}

for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

for name, path in DIRS.items():
    print(f"{name}: {path}")

print("\nDirectory structure ready.")

In [5]:
from pathlib import Path

BASE_DIR = Path("/kaggle/working/geo_data")

DIRS = {
    "boundaries": BASE_DIR / "boundaries",
    "schools_raw": BASE_DIR / "schools_raw",
    "schools_processed": BASE_DIR / "schools_processed",
    "population": BASE_DIR / "population",
    "outputs": BASE_DIR / "outputs",
    "logs": BASE_DIR / "logs"
}

for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

for name, path in DIRS.items():
    print(f"{name}: {path}")

print("\nDirectory structure ready.")

boundaries: /kaggle/working/geo_data/boundaries
schools_raw: /kaggle/working/geo_data/schools_raw
schools_processed: /kaggle/working/geo_data/schools_processed
population: /kaggle/working/geo_data/population
outputs: /kaggle/working/geo_data/outputs
logs: /kaggle/working/geo_data/logs

Directory structure ready.


In [6]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/vedantsaraswate/udise-2526/enrolment_data_2_All State_2025-26/100_enr2.csv
/kaggle/input/datasets/vedantsaraswate/udise-2526/facility_data_All State_2025-26/100_fac.csv
/kaggle/input/datasets/vedantsaraswate/udise-2526/profile_data_1_All State_2025-26/100_prof1.csv
/kaggle/input/datasets/vedantsaraswate/udise-2526/profile_data_2_All State_2025-26/100_prof2.csv
/kaggle/input/datasets/vedantsaraswate/udise-2526/teacher_data_All State_2025-26/100_tch.csv


In [7]:

from pathlib import Path

BASE = Path(
    "/kaggle/input/datasets/vedantsaraswate/udise-2526"
)

FILES = {
    "enrolment": BASE / "enrolment_data_2_All State_2025-26/100_enr2.csv",
    "facility": BASE / "facility_data_All State_2025-26/100_fac.csv",
    "profile1": BASE / "profile_data_1_All State_2025-26/100_prof1.csv",
    "profile2": BASE / "profile_data_2_All State_2025-26/100_prof2.csv",
    "teacher": BASE / "teacher_data_All State_2025-26/100_tch.csv"
}

for name, path in FILES.items():
    print(
        f"{name}:",
        "FOUND" if path.exists() else "MISSING"
    )

print("\nUDISE dataset paths configured.")


enrolment: FOUND
facility: FOUND
profile1: FOUND
profile2: FOUND
teacher: FOUND

UDISE dataset paths configured.


In [8]:

import requests
import geopandas as gpd
from io import BytesIO

# Source: Pune 2022 ward polygons
URL = (
    "https://raw.githubusercontent.com/"
    "answerquest/pune-2022-wards/main/pune-2022-wards.geojson"
)

response = requests.get(URL, timeout=90)
response.raise_for_status()

# Load original ward polygons
pune_wards = gpd.read_file(BytesIO(response.content))
pune_wards = pune_wards.to_crs("EPSG:4326")

# Save raw ward data immediately
WARD_PATH = DIRS["boundaries"] / "pune_wards_2022.geojson"
pune_wards.to_file(WARD_PATH, driver="GeoJSON")

# Merge wards into one municipal boundary
pune_boundary_geom = pune_wards.geometry.union_all()

pune_boundary = gpd.GeoDataFrame(
    {"city": ["Pune"], "boundary_year": [2022]},
    geometry=[pune_boundary_geom],
    crs="EPSG:4326"
)

# Validate
area_km2 = (
    pune_boundary.to_crs("EPSG:32643").geometry.area.iloc[0]
    / 1_000_000
)

print("Ward features:", len(pune_wards))
print("Geometry:", pune_boundary_geom.geom_type)
print("Valid:", pune_boundary_geom.is_valid)
print("Area (km²):", round(area_km2, 2))
print("Bounds:", pune_boundary.total_bounds)

# Save merged boundary
BOUNDARY_PATH = DIRS["boundaries"] / "pune_boundary.geojson"
pune_boundary.to_file(BOUNDARY_PATH, driver="GeoJSON")

print("\nCHECKPOINTS SAVED:")
print(WARD_PATH)
print(BOUNDARY_PATH)


Ward features: 58
Geometry: Polygon
Valid: True
Area (km²): 506.91
Bounds: [73.73193 18.38539 74.01838 18.62185]

CHECKPOINTS SAVED:
/kaggle/working/geo_data/boundaries/pune_wards_2022.geojson
/kaggle/working/geo_data/boundaries/pune_boundary.geojson


In [9]:

import requests
import pandas as pd

REPO = "yashveeeeeeer/india-geodata"
TREE_URL = (
    f"https://api.github.com/repos/{REPO}/git/trees/main"
    "?recursive=1"
)

cities = {
    "Pune": ["pune"],
    "Mumbai": ["mumbai"],
    "Hyderabad": ["hyderabad"],
    "Bengaluru": ["bangalore", "bengaluru"],
    "Chennai": ["chennai"],
    "Kolkata": ["kolkata"],
    "Surat": ["surat"],
    "Ahmedabad": ["ahmedabad"],
    "Jaipur": ["jaipur"],
    "Lucknow": ["lucknow"],
    "Kozhikode": ["kozhikode", "calicut"]
}

response = requests.get(TREE_URL, timeout=120)
response.raise_for_status()

tree = response.json()["tree"]

base = "data/urban/municipal-boundaries/"

files = [
    item["path"]
    for item in tree
    if item["type"] == "blob"
    and item["path"].startswith(base)
]

records = []

for city, aliases in cities.items():
    candidates = []

    for path in files:
        parts = path.lower().split("/")
        if len(parts) < 4:
            continue

        folder = parts[3]

        if folder in aliases and path.lower().endswith(
            (".geojson", ".json", ".kml", ".zip")
        ):
            candidates.append(path)

    for path in candidates:
        records.append({
            "city": city,
            "candidate_file": path,
            "format": path.rsplit(".", 1)[-1]
        })

manifest = pd.DataFrame(records)

for city in cities:
    print("\n" + "=" * 50)
    print(city)

    matches = manifest[manifest["city"] == city]

    if matches.empty:
        print("No candidate files found in this repository.")
    else:
        for path in matches["candidate_file"]:
            print(" -", path)

print("\nDISCOVERY SUMMARY")
print("Cities:", len(cities))
print("Cities with candidate files:",
      manifest["city"].nunique())
print("Total candidate files:", len(manifest))



Pune
 - data/urban/municipal-boundaries/pune/pune-admin-wards_2017.geojson
 - data/urban/municipal-boundaries/pune/pune-electoral-wards_2012.geojson
 - data/urban/municipal-boundaries/pune/pune-electoral-wards_2017.geojson
 - data/urban/municipal-boundaries/pune/pune-electoral-wards_2022.geojson

Mumbai
 - data/urban/municipal-boundaries/mumbai/BMC_Wards.geojson
 - data/urban/municipal-boundaries/mumbai/BMC_Wards.kml
 - data/urban/municipal-boundaries/mumbai/slumClusters.geojson

Hyderabad
 - data/urban/municipal-boundaries/hyderabad/ghmc-area.geojson
 - data/urban/municipal-boundaries/hyderabad/ghmc-area.kml
 - data/urban/municipal-boundaries/hyderabad/ghmc-wards.geojson
 - data/urban/municipal-boundaries/hyderabad/ghmc-wards.kml
 - data/urban/municipal-boundaries/hyderabad/ghmc-zones.geojson
 - data/urban/municipal-boundaries/hyderabad/ghmc-zones.kml

Bengaluru
 - data/urban/municipal-boundaries/bangalore/BBMP.geojson
 - data/urban/municipal-boundaries/bangalore/BBMP_oldWards.geojso

In [10]:

import requests
import geopandas as gpd
import pandas as pd
from pathlib import Path
from io import BytesIO

REPO = "yashveeeeeeer/india-geodata"
BRANCH = "main"

RAW_DIR = DIRS["boundaries"] / "raw_sources"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Selected candidate files from repository discovery
sources = {
    "Mumbai": "data/urban/municipal-boundaries/mumbai/BMC_Wards.geojson",
    "Hyderabad": "data/urban/municipal-boundaries/hyderabad/ghmc-area.geojson",
    "Bengaluru": "data/urban/municipal-boundaries/bangalore/BBMP.geojson",
    "Chennai": "data/urban/municipal-boundaries/chennai/Wards.geojson",
    "Kolkata": "data/urban/municipal-boundaries/kolkata/kolkata.geojson",
    "Ahmedabad": "data/urban/municipal-boundaries/ahmedabad/Wards.geojson",
    "Jaipur": "data/urban/municipal-boundaries/jaipur/Jaipur_Boundary.geojson",
    "Lucknow": "data/urban/municipal-boundaries/lucknow/Lucknow_city_boundary.geojson"
}

headers = {"User-Agent": "HiMCM-Education-Index/1.0"}

downloaded = {}
report = []

for city, repo_path in sources.items():

    print("\nDownloading:", city)

    url = (
        f"https://raw.githubusercontent.com/"
        f"{REPO}/{BRANCH}/{repo_path}"
    )

    response = requests.get(
        url, headers=headers, timeout=120
    )
    response.raise_for_status()

    local_path = RAW_DIR / f"{city.lower()}_source.geojson"
    local_path.write_bytes(response.content)

    gdf = gpd.read_file(BytesIO(response.content))

    if gdf.crs is None:
        print("WARNING: CRS is missing.")
        report.append({
            "city": city,
            "source": repo_path,
            "status": "CRS missing",
            "features": len(gdf)
        })
        continue

    gdf = gdf.to_crs("EPSG:4326")
    downloaded[city] = gdf

    # Approximate area using equal-area projection
    area_km2 = (
        gdf.to_crs("EPSG:6933").geometry.area.sum()
        / 1_000_000
    )

    geom_types = gdf.geom_type.value_counts().to_dict()

    report.append({
        "city": city,
        "source": repo_path,
        "status": "Downloaded",
        "features": len(gdf),
        "geometry_types": str(geom_types),
        "all_valid": bool(gdf.geometry.is_valid.all()),
        "area_sum_km2": round(area_km2, 2),
        "bounds": str(gdf.total_bounds),
        "crs": str(gdf.crs)
    })

    print("Features:", len(gdf))
    print("Geometry types:", geom_types)
    print("All geometries valid:", gdf.geometry.is_valid.all())
    print("Area sum (km²):", round(area_km2, 2))
    print("Saved:", local_path)

report_df = pd.DataFrame(report)

print("\n" + "=" * 65)
print("BOUNDARY SOURCE VALIDATION REPORT")
print("=" * 65)

display(report_df)

report_df.to_csv(
    DIRS["logs"] / "boundary_source_validation.csv",
    index=False
)

print("\nValidation report saved.")
print("Cities still pending: Surat, Kozhikode.")



Downloading: Mumbai
Features: 24
Geometry types: {'MultiPolygon': 24}
All geometries valid: True
Area sum (km²): 458.08
Saved: /kaggle/working/geo_data/boundaries/raw_sources/mumbai_source.geojson

Downloading: Hyderabad
Features: 1
Geometry types: {'MultiPolygon': 1}
All geometries valid: True
Area sum (km²): 608.87
Saved: /kaggle/working/geo_data/boundaries/raw_sources/hyderabad_source.geojson

Downloading: Bengaluru
Features: 243
Geometry types: {'Polygon': 243}
All geometries valid: False
Area sum (km²): 703.27
Saved: /kaggle/working/geo_data/boundaries/raw_sources/bengaluru_source.geojson

Downloading: Chennai
Features: 201
Geometry types: {'Polygon': 201}
All geometries valid: True
Area sum (km²): 434.6
Saved: /kaggle/working/geo_data/boundaries/raw_sources/chennai_source.geojson

Downloading: Kolkata
Features: 141
Geometry types: {'MultiPolygon': 141}
All geometries valid: False
Area sum (km²): 186.63
Saved: /kaggle/working/geo_data/boundaries/raw_sources/kolkata_source.geojson

,city,source,status,features,geometry_types,all_valid,area_sum_km2,bounds,crs
0,Mumbai,data/urban/municipal-boundaries/mumbai/BMC_War...,Downloaded,24,{'MultiPolygon': 24},True,458.08,[72.77633295 18.89395643 72.9797315 19.27017667],EPSG:4326
1,Hyderabad,data/urban/municipal-boundaries/hyderabad/ghmc...,Downloaded,1,{'MultiPolygon': 1},True,608.87,[78.2387067 17.2916377 78.6223912 17.5608321],EPSG:4326
2,Bengaluru,data/urban/municipal-boundaries/bangalore/BBMP...,Downloaded,243,{'Polygon': 243},False,703.27,[77.4598797 12.8334905 77.7843613 13.1426195],EPSG:4326
3,Chennai,data/urban/municipal-boundaries/chennai/Wards....,Downloaded,201,{'Polygon': 201},True,434.60,[80.14017661 12.85197253 80.33122353 13.23518545],EPSG:4326
4,Kolkata,data/urban/municipal-boundaries/kolkata/kolkat...,Downloaded,141,{'MultiPolygon': 141},False,186.63,[88.24214287 22.45032345 88.45895489 22.63257693],EPSG:4326
5,Ahmedabad,data/urban/municipal-boundaries/ahmedabad/Ward...,Downloaded,48,{'Polygon': 48},True,440.74,[72.449281 22.9139487 72.7014909 23.140458 ],EPSG:4326
6,Jaipur,data/urban/municipal-boundaries/jaipur/Jaipur_...,Downloaded,1,{'Polygon': 1},True,376.22,[75.6887556 26.77468664 75.91181677 27.02186617],EPSG:4326
7,Lucknow,data/urban/municipal-boundaries/lucknow/Luckno...,Downloaded,1,{'Polygon': 1},True,283.58,[80.83109058 26.74407589 81.06106184 26.93432069],EPSG:4326



Validation report saved.
Cities still pending: Surat, Kozhikode.


In [11]:

import geopandas as gpd
import pandas as pd
import shapely

from shapely import make_valid, is_valid_reason

diagnostics = []
repaired_boundaries = {}

for city, gdf in downloaded.items():

    print("\nProcessing:", city)

    # Reproject to an equal-area CRS for area calculations
    projected = gdf.to_crs("EPSG:6933")

    raw_areas = projected.geometry.area
    invalid_mask = ~projected.geometry.is_valid

    invalid_reasons = (
        projected.loc[invalid_mask, "geometry"]
        .apply(is_valid_reason)
        .value_counts()
        .to_dict()
    )

    # Repair individual geometries for diagnostic purposes
    repaired_geom = make_valid(projected.geometry.array)

    repaired_gdf = gpd.GeoDataFrame(
        geometry=repaired_geom,
        crs="EPSG:6933"
    )

    # Dissolve all features into one actual city footprint
    city_union = repaired_gdf.geometry.union_all()

    sum_area = repaired_gdf.geometry.area.sum() / 1e6
    union_area = city_union.area / 1e6

    overlap_excess = sum_area - union_area

    # Convert final geometry back to WGS84
    final_geometry = gpd.GeoSeries(
        [city_union], crs="EPSG:6933"
    ).to_crs("EPSG:4326").iloc[0]

    repaired_boundaries[city] = final_geometry

    diagnostics.append({
        "city": city,
        "features": len(gdf),
        "invalid_features": int(invalid_mask.sum()),
        "invalid_reasons": str(invalid_reasons),
        "area_sum_km2": round(sum_area, 2),
        "union_area_km2": round(union_area, 2),
        "overlap_excess_km2": round(overlap_excess, 2),
        "repaired_union_valid": bool(city_union.is_valid),
        "union_geometry_type": city_union.geom_type
    })

    print("Invalid features:", int(invalid_mask.sum()))
    print("Invalidity reasons:", invalid_reasons)
    print("Sum of feature areas:", round(sum_area, 2), "km²")
    print("Actual union area:", round(union_area, 2), "km²")
    print("Overlap excess:", round(overlap_excess, 2), "km²")
    print("Repaired union valid:", city_union.is_valid)

diagnostic_df = pd.DataFrame(diagnostics)

display(diagnostic_df)

diagnostic_df.to_csv(
    DIRS["logs"] / "geometry_diagnostics.csv",
    index=False
)

print("\nDiagnostic report saved.")



Processing: Mumbai
Invalid features: 0
Invalidity reasons: {}
Sum of feature areas: 458.08 km²
Actual union area: 458.08 km²
Overlap excess: -0.0 km²
Repaired union valid: True

Processing: Hyderabad
Invalid features: 0
Invalidity reasons: {}
Sum of feature areas: 608.87 km²
Actual union area: 608.87 km²
Overlap excess: 0.0 km²
Repaired union valid: True

Processing: Bengaluru
Invalid features: 6
Invalidity reasons: {'Self-intersection[7486835.73125656 1642380.23409542]': 2, 'Hole lies outside shell[7490179.52179566 1658355.28783786]': 1, 'Hole lies outside shell[7478551.54527162 1636308.8229053]': 1, 'Hole lies outside shell[7492782.86636625 1653846.08885403]': 1, 'Hole lies outside shell[7482319.84589271 1633609.51897973]': 1}
Sum of feature areas: 716.28 km²
Actual union area: 716.28 km²
Overlap excess: 0.0 km²
Repaired union valid: True

Processing: Chennai
Invalid features: 0
Invalidity reasons: {}
Sum of feature areas: 434.6 km²
Actual union area: 434.6 km²
Overlap excess: 0.0 k

,city,features,invalid_features,invalid_reasons,area_sum_km2,union_area_km2,overlap_excess_km2,repaired_union_valid,union_geometry_type
0,Mumbai,24,0,{},458.08,458.08,-0.00,True,MultiPolygon
1,Hyderabad,1,0,{},608.87,608.87,0.00,True,MultiPolygon
2,Bengaluru,243,6,{'Self-intersection[7486835.73125656 1642380.2...,716.28,716.28,0.00,True,Polygon
3,Chennai,201,0,{},434.60,434.60,0.00,True,Polygon
4,Kolkata,141,1,{'Ring Self-intersection[8526937.40299159 2806...,186.63,186.61,0.02,True,MultiPolygon
5,Ahmedabad,48,0,{},440.74,439.47,1.27,True,Polygon
6,Jaipur,1,0,{},376.22,376.22,0.00,True,Polygon
7,Lucknow,1,0,{},283.58,283.58,0.00,True,Polygon



Diagnostic report saved.


In [12]:

import geopandas as gpd
import pandas as pd

# ------------------------------------------------------------
# BENGALURU: compare original and repaired ward areas
# ------------------------------------------------------------

print("=" * 65)
print("BENGALURU — GEOMETRY REPAIR IMPACT")
print("=" * 65)

blr = downloaded["Bengaluru"].to_crs("EPSG:6933")

blr_original_area = blr.geometry.area / 1e6
blr_repaired_geom = shapely.make_valid(blr.geometry.array)
blr_repaired_area = pd.Series(
    [g.area / 1e6 for g in blr_repaired_geom],
    index=blr.index
)

blr_report = pd.DataFrame({
    "feature_index": blr.index,
    "original_area_km2": blr_original_area,
    "repaired_area_km2": blr_repaired_area
})

blr_report["area_change_km2"] = (
    blr_report["repaired_area_km2"]
    - blr_report["original_area_km2"]
)

invalid_indices = blr.index[~blr.geometry.is_valid]

print("Invalid ward indices:", list(invalid_indices))
print("\nChanges in invalid wards:")

display(
    blr_report.loc[invalid_indices]
    .sort_values("area_change_km2", ascending=False)
)

print(
    "\nTotal area change:",
    round(blr_report["area_change_km2"].sum(), 4),
    "km²"
)

# ------------------------------------------------------------
# AHMEDABAD: identify overlapping ward pairs
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("AHMEDABAD — WARD OVERLAP INSPECTION")
print("=" * 65)

amd = downloaded["Ahmedabad"].to_crs("EPSG:6933")

overlaps = []

for i in range(len(amd)):
    geom_i = amd.geometry.iloc[i]

    for j in range(i + 1, len(amd)):
        geom_j = amd.geometry.iloc[j]

        if geom_i.intersects(geom_j):
            intersection = geom_i.intersection(geom_j)
            overlap_area = intersection.area / 1e6

            if overlap_area > 0.001:
                overlaps.append({
                    "ward_index_1": amd.index[i],
                    "ward_index_2": amd.index[j],
                    "overlap_km2": overlap_area
                })

overlap_df = pd.DataFrame(overlaps)

if overlap_df.empty:
    print("No material ward overlaps detected.")
else:
    overlap_df = overlap_df.sort_values(
        "overlap_km2", ascending=False
    )
    display(overlap_df)

print("\nTotal pairwise overlap area:",
      round(overlap_df["overlap_km2"].sum(), 4)
      if not overlap_df.empty else 0,
      "km²")


BENGALURU — GEOMETRY REPAIR IMPACT
Invalid ward indices: [0, 49, 60, 175, 188, 224]

Changes in invalid wards:


,feature_index,original_area_km2,repaired_area_km2,area_change_km2
175,175,2.943184,14.429204,1.148602e+01
0,0,10.339286,11.359773,1.020487e+00
49,49,6.392844,6.771446,3.786024e-01
224,224,1.200964,1.334173,1.332098e-01
188,188,4.344436,4.344436,7.529088e-12
60,60,3.541623,3.541623,-7.529533e-12



Total area change: 13.0183 km²

AHMEDABAD — WARD OVERLAP INSPECTION


,ward_index_1,ward_index_2,overlap_km2
6,2,14,0.377334
0,0,1,0.147701
40,31,38,0.108994
24,19,30,0.090043
22,18,19,0.077663
21,13,18,0.069392
39,30,31,0.040092
36,26,35,0.032456
20,13,17,0.029948
30,22,26,0.028988



Total pairwise overlap area: 1.2623 km²


In [13]:

import pandas as pd
import geopandas as gpd
import shapely

# ============================================================
# BENGALURU — inspect invalid wards
# ============================================================

blr = downloaded["Bengaluru"].to_crs("EPSG:6933")

invalid_indices = [0, 49, 60, 175, 188, 224]

print("=" * 65)
print("BENGALURU — AFFECTED WARDS")
print("=" * 65)

for idx in invalid_indices:

    row = blr.loc[idx]
    original = row.geometry
    repaired = shapely.make_valid(original)

    print("\nWARD INDEX:", idx)

    attrs = row.drop(labels="geometry")
    print("Attributes:")
    print(attrs.to_dict())

    print("Original area:", round(original.area / 1e6, 4), "km²")
    print("Original validity:", original.is_valid)
    print("Original reason:", shapely.is_valid_reason(original))
    print("Original geometry type:", original.geom_type)

    print("Repaired area:", round(repaired.area / 1e6, 4), "km²")
    print("Repaired geometry type:", repaired.geom_type)

    if repaired.geom_type == "GeometryCollection":
        print("Repaired components:")
        for part in repaired.geoms:
            print(part.geom_type, round(part.area / 1e6, 4), "km²")


# ============================================================
# AHMEDABAD — inspect the largest overlapping pairs
# ============================================================

amd = downloaded["Ahmedabad"].to_crs("EPSG:6933")

print("\n" + "=" * 65)
print("AHMEDABAD — LARGEST OVERLAPPING WARD PAIRS")
print("=" * 65)

for i, j in [(2, 14), (0, 1), (31, 38), (19, 30), (18, 19)]:

    a = amd.iloc[i]
    b = amd.iloc[j]

    print(f"\nFEATURES {i} AND {j}")

    print("Feature A:", a.drop(labels="geometry").to_dict())
    print("Feature B:", b.drop(labels="geometry").to_dict())

    overlap = a.geometry.intersection(b.geometry)

    print("Overlap area:", round(overlap.area / 1e6, 5), "km²")
    print("Overlap geometry:", overlap.geom_type)


BENGALURU — AFFECTED WARDS

WARD INDEX: 0
Attributes:
{'KGISWardID': 4878, 'KGISWardCode': '2003001', 'LGD_WardCode': 1303139.0, 'KGISWardNo': '1', 'KGISWardName': 'Kempegowda Ward', 'KGISTownCode': '2003'}
Original area: 10.3393 km²
Original validity: False
Original reason: Hole lies outside shell[7490179.52179566 1658355.28783786]
Original geometry type: Polygon
Repaired area: 11.3598 km²
Repaired geometry type: MultiPolygon

WARD INDEX: 49
Attributes:
{'KGISWardID': 4943, 'KGISWardCode': '2003034', 'LGD_WardCode': 1302706.0, 'KGISWardNo': '34', 'KGISWardName': 'Bande Mutt', 'KGISTownCode': '2003'}
Original area: 6.3928 km²
Original validity: False
Original reason: Hole lies outside shell[7478551.54527162 1636308.8229053]
Original geometry type: Polygon
Repaired area: 6.7714 km²
Repaired geometry type: MultiPolygon

WARD INDEX: 60
Attributes:
{'KGISWardID': 5134, 'KGISWardCode': '2003129', 'LGD_WardCode': 1302801.0, 'KGISWardNo': '129', 'KGISWardName': 'Sampangiram Nagar', 'KGISTownC

In [14]:

import requests
import geopandas as gpd
from io import BytesIO

# OpenCity Ahmedabad 2024 ward dataset
API = "https://data.opencity.in/api/3/action/package_show"

r = requests.get(
    API,
    params={"id": "amdavad-municipal-corporation-wards-map-2024"},
    timeout=90
)

print("API status:", r.status_code)

if r.ok:
    package = r.json()["result"]

    for resource in package["resources"]:
        print(
            resource.get("name"),
            resource.get("format"),
            resource.get("url")
        )
else:
    print("OpenCity API unavailable:", r.text[:500])


API status: 200
Amdavad Wards Map 2024 KML https://data.opencity.in/dataset/f7505ef4-810a-4119-a649-8e8fb1287e5e/resource/d251953b-6168-4c73-a496-40e53ffb9349/download/6a6d025b-2b02-4b0e-8bb8-74186eacf071.kml
Amdavad Wards Map 2024 KML https://data.opencity.in/dataset/f7505ef4-810a-4119-a649-8e8fb1287e5e/resource/414862b3-b382-4692-963d-38f6ce842483/download/6a6d025b-2b02-4b0e-8bb8-74186eacf071.kml


In [15]:

import requests
import hashlib
import geopandas as gpd
import fiona

from pathlib import Path
from io import BytesIO

URLS = [
    "https://data.opencity.in/dataset/f7505ef4-810a-4119-a649-8e8fb1287e5e/resource/d251953b-6168-4c73-a496-40e53ffb9349/download/6a6d025b-2b02-4b0e-8bb8-74186eacf071.kml",

    "https://data.opencity.in/dataset/f7505ef4-810a-4119-a649-8e8fb1287e5e/resource/414862b3-b382-4692-963d-38f6ce842483/download/6a6d025b-2b02-4b0e-8bb8-74186eacf071.kml"
]

KML_DIR = DIRS["boundaries"] / "raw_sources"
KML_DIR.mkdir(parents=True, exist_ok=True)

headers = {"User-Agent": "HiMCM-Education-Index/1.0"}

downloads = []

for i, url in enumerate(URLS, start=1):
    response = requests.get(
        url, headers=headers, timeout=120
    )
    response.raise_for_status()

    content = response.content

    sha = hashlib.sha256(content).hexdigest()

    downloads.append({
        "resource": i,
        "content": content,
        "sha256": sha,
        "size": len(content)
    })

    print(f"Resource {i}:")
    print("HTTP status:", response.status_code)
    print("File size:", len(content))
    print("SHA256:", sha)

print("\nIdentical files:",
      downloads[0]["sha256"] == downloads[1]["sha256"])

# Preserve the first resource as the raw source
KML_PATH = KML_DIR / "ahmedabad_wards_2024.kml"
KML_PATH.write_bytes(downloads[0]["content"])

# Read KML
fiona.drvsupport.supported_drivers["KML"] = "rw"

amd_2024 = gpd.read_file(
    BytesIO(downloads[0]["content"]),
    driver="KML"
)

print("\nAHMEDABAD 2024 SOURCE")
print("Features:", len(amd_2024))
print("Columns:", amd_2024.columns.tolist())
print("Geometry types:", amd_2024.geom_type.value_counts().to_dict())
print("CRS:", amd_2024.crs)

if amd_2024.crs is None:
    raise ValueError("KML has no recognised CRS.")

amd_2024 = amd_2024.to_crs("EPSG:4326")

print("All geometries valid:",
      amd_2024.geometry.is_valid.all())

print("Bounds:", amd_2024.total_bounds)

area_km2 = (
    amd_2024.to_crs("EPSG:6933").geometry.union_all().area
    / 1_000_000
)

print("Union area (km²):", round(area_km2, 2))

# Save candidate separately
AMD_CANDIDATE_PATH = (
    DIRS["boundaries"] / "ahmedabad_wards_2024_candidate.geojson"
)

amd_2024.to_file(
    AMD_CANDIDATE_PATH,
    driver="GeoJSON"
)

print("\nRaw KML saved:", KML_PATH)
print("Candidate GeoJSON saved:", AMD_CANDIDATE_PATH)


Resource 1:
HTTP status: 200
File size: 293763
SHA256: 620f31cbcafb355a7b74b70a9a1fdcf91416dc28377273d41785e700ec8b7bbd
Resource 2:
HTTP status: 200
File size: 293763
SHA256: 620f31cbcafb355a7b74b70a9a1fdcf91416dc28377273d41785e700ec8b7bbd

Identical files: True

AHMEDABAD 2024 SOURCE
Features: 48
Columns: ['id', 'Name', 'description', 'timestamp', 'begin', 'end', 'altitudeMode', 'tessellate', 'extrude', 'visibility', 'drawOrder', 'icon', 'objectid', 'townname', 'towncensuscode2011', 'town_lgd_code', 'ward_lgd_code', 'ward_lgd_name', 'sourcewardname', 'sourcewardcode', 'state', 'geometry']
Geometry types: {'MultiPolygon': 48}
CRS: EPSG:4326
All geometries valid: True
Bounds: [72.44724336 22.91214073 72.70369455 23.13864753]
Union area (km²): 438.44

Raw KML saved: /kaggle/working/geo_data/boundaries/raw_sources/ahmedabad_wards_2024.kml
Candidate GeoJSON saved: /kaggle/working/geo_data/boundaries/ahmedabad_wards_2024_candidate.geojson


In [16]:

# ============================================================
# HiMCM — FINAL 10-CITY MUNICIPAL BOUNDARIES
# Acquisition, repair, validation and export
# ============================================================

import os
import requests
import geopandas as gpd
import pandas as pd
import shapely
import fiona

from pathlib import Path
from io import BytesIO

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "yashveeeeeeer/india-geodata/main/"
    "data/urban/municipal-boundaries/"
)

BOUNDARY_DIR = DIRS["boundaries"]
RAW_DIR = BOUNDARY_DIR / "raw_sources"
FINAL_DIR = BOUNDARY_DIR / "final"

RAW_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {
    "User-Agent": "HiMCM-Education-Index/1.0"
}

# ------------------------------------------------------------
# 1. LOAD OR DOWNLOAD SOURCE FILES
# ------------------------------------------------------------

def load_geojson(city, url, local_path=None):

    if local_path and Path(local_path).exists():
        print(city, "- loading existing file")
        return gpd.read_file(local_path)

    print(city, "- downloading GeoJSON")

    r = requests.get(url, headers=HEADERS, timeout=120)
    r.raise_for_status()

    path = RAW_DIR / f"{city.lower()}_source.geojson"
    path.write_bytes(r.content)

    return gpd.read_file(BytesIO(r.content))


# ------------------------------------------------------------
# 2. LOAD PUNE
# ------------------------------------------------------------

pune_path = BOUNDARY_DIR / "pune_boundary.geojson"

if not pune_path.exists():
    raise FileNotFoundError(
        "Pune boundary checkpoint missing. Recreate it first."
    )

sources = {
    "Pune": (
        gpd.read_file(pune_path),
        "Pune 2022 ward union",
        "Ward union"
    )
}


# ------------------------------------------------------------
# 3. LOAD EXISTING AND REMAINING CITIES
# ------------------------------------------------------------

city_sources = {
    "Mumbai": {
        "url": BASE_URL + "mumbai/BMC_Wards.geojson",
        "local": RAW_DIR / "mumbai_source.geojson",
        "name": "BMC_Wards.geojson",
        "type": "Ward union"
    },

    "Hyderabad": {
        "url": BASE_URL + "hyderabad/ghmc-area.geojson",
        "local": RAW_DIR / "hyderabad_source.geojson",
        "name": "ghmc-area.geojson",
        "type": "Direct municipal boundary"
    },

    "Chennai": {
        "url": BASE_URL + "chennai/Wards.geojson",
        "local": RAW_DIR / "chennai_source.geojson",
        "name": "Wards.geojson",
        "type": "Ward union"
    },

    "Kolkata": {
        "url": BASE_URL + "kolkata/kolkata.geojson",
        "local": RAW_DIR / "kolkata_source.geojson",
        "name": "kolkata.geojson",
        "type": "Ward union"
    },

    "Ahmedabad": {
        "url": None,
        "local": BOUNDARY_DIR / "ahmedabad_wards_2024_candidate.geojson",
        "name": "Amdavad Wards Map 2024",
        "type": "Ward union"
    },

    "Jaipur": {
        "url": BASE_URL + "jaipur/Jaipur_Boundary.geojson",
        "local": RAW_DIR / "jaipur_source.geojson",
        "name": "Jaipur_Boundary.geojson",
        "type": "Direct municipal boundary"
    },

    "Lucknow": {
        "url": BASE_URL + "lucknow/Lucknow_city_boundary.geojson",
        "local": RAW_DIR / "lucknow_source.geojson",
        "name": "Lucknow_city_boundary.geojson",
        "type": "Direct municipal boundary"
    },

    "Bhubaneswar": {
        "url": BASE_URL + "bhubaneswar/BMC%20Boundary.GeoJSON",
        "local": None,
        "name": "BMC Boundary.GeoJSON",
        "type": "Direct municipal boundary"
    }
}

for city, info in city_sources.items():

    if info["local"] and Path(info["local"]).exists():
        gdf = gpd.read_file(info["local"])

    elif info["url"]:
        gdf = load_geojson(
            city,
            info["url"],
            info["local"]
        )

    else:
        raise FileNotFoundError(
            f"Missing source file for {city}: {info['local']}"
        )

    sources[city] = (
        gdf,
        info["name"],
        info["type"]
    )


# ------------------------------------------------------------
# 4. DOWNLOAD INDORE 2024 — 85 WARDS
# ------------------------------------------------------------

print("\nAcquiring Indore 2024 wards...")

api = "https://data.opencity.in/api/3/action/package_show"

r = requests.get(
    api,
    params={"id": "indore-wards-map"},
    timeout=90
)
r.raise_for_status()

package = r.json()["result"]

resource = next(
    x for x in package["resources"]
    if x.get("format", "").upper() == "KML"
)

indore_url = resource["url"]

r = requests.get(
    indore_url,
    headers=HEADERS,
    timeout=120
)
r.raise_for_status()

indore_raw_path = RAW_DIR / "indore_wards_2024.kml"
indore_raw_path.write_bytes(r.content)

fiona.drvsupport.supported_drivers["KML"] = "rw"

indore = gpd.read_file(
    BytesIO(r.content),
    driver="KML"
)

sources["Indore"] = (
    indore,
    "Indore Wards Map 2024 — 85 wards",
    "Ward union"
)

print("Indore features:", len(indore))


# ------------------------------------------------------------
# 5. NORMALISE, REPAIR AND DISSOLVE
# ------------------------------------------------------------

final_boundaries = []
validation_records = []

for city, (gdf, source_name, source_type) in sources.items():

    print("\nProcessing:", city)

    if gdf.empty:
        raise ValueError(f"{city}: source dataset is empty.")

    if gdf.crs is None:
        raise ValueError(f"{city}: source CRS is missing.")

    # Use WGS84 for storage and EPSG:6933 for area calculations
    projected = gdf.to_crs("EPSG:6933")

    original_area = projected.geometry.area.sum() / 1e6

    invalid_count = int(
        (~projected.geometry.is_valid).sum()
    )

    # Repair invalid features, retaining polygonal components
    repaired = shapely.make_valid(
        projected.geometry.array
    )

    repaired_gdf = gpd.GeoDataFrame(
        geometry=repaired,
        crs="EPSG:6933"
    )

    # Dissolve all source features into one city footprint
    union_geom = repaired_gdf.geometry.union_all()

    if union_geom.is_empty:
        raise ValueError(f"{city}: dissolved geometry is empty.")

    if not union_geom.is_valid:
        raise ValueError(
            f"{city}: final geometry remains invalid."
        )

    union_area = union_geom.area / 1e6

    overlap_excess = (
        repaired_gdf.geometry.area.sum() / 1e6
        - union_area
    )

    repair_change = union_area - original_area

    # Return to WGS84
    final_geom = gpd.GeoSeries(
        [union_geom],
        crs="EPSG:6933"
    ).to_crs("EPSG:4326").iloc[0]

    boundary = gpd.GeoDataFrame(
        {
            "city": [city],
            "source": [source_name],
            "source_type": [source_type],
            "source_features": [len(gdf)],
            "invalid_source_features": [invalid_count],
            "area_km2": [round(union_area, 4)]
        },
        geometry=[final_geom],
        crs="EPSG:4326"
    )

    # Save each accepted candidate separately
    output_path = FINAL_DIR / f"{city.lower()}_boundary.geojson"

    boundary.to_file(
        output_path,
        driver="GeoJSON"
    )

    final_boundaries.append(boundary)

    validation_records.append({
        "city": city,
        "source": source_name,
        "source_type": source_type,
        "features": len(gdf),
        "invalid_features_repaired": invalid_count,
        "original_area_sum_km2": round(original_area, 4),
        "final_union_area_km2": round(union_area, 4),
        "repair_area_change_km2": round(repair_change, 4),
        "overlap_excess_km2": round(overlap_excess, 4),
        "final_geometry_valid": final_geom.is_valid,
        "final_geometry_type": final_geom.geom_type,
        "saved_path": str(output_path)
    })

    print("Features:", len(gdf))
    print("Invalid source features:", invalid_count)
    print("Union area:", round(union_area, 2), "km²")
    print("Overlap excess:", round(overlap_excess, 3), "km²")
    print("Final geometry valid:", final_geom.is_valid)


# ------------------------------------------------------------
# 6. COMBINE ALL 10 CITIES
# ------------------------------------------------------------

combined = gpd.GeoDataFrame(
    pd.concat(final_boundaries, ignore_index=True),
    geometry="geometry",
    crs="EPSG:4326"
)

if len(combined) != 10:
    raise ValueError(
        f"Expected 10 cities, obtained {len(combined)}."
    )

if not combined["city"].is_unique:
    raise ValueError("Duplicate city names detected.")

combined_path = FINAL_DIR / "final_10_city_boundaries.geojson"

combined.to_file(
    combined_path,
    driver="GeoJSON"
)

# Save validation manifest
validation = pd.DataFrame(validation_records)

manifest_path = FINAL_DIR / "boundary_validation_manifest.csv"

validation.to_csv(
    manifest_path,
    index=False
)

print("\n" + "=" * 70)
print("FINAL 10-CITY BOUNDARY ACQUISITION")
print("=" * 70)

display(validation)

print("\nCombined boundary file:", combined_path)
print("Validation manifest:", manifest_path)
print("Total cities:", len(combined))
print("All final geometries valid:",
      combined.geometry.is_valid.all())

print("\nBOUNDARY STAGE COMPLETE.")


Bhubaneswar - downloading GeoJSON

Acquiring Indore 2024 wards...
Indore features: 85

Processing: Pune
Features: 1
Invalid source features: 0
Union area: 507.13 km²
Overlap excess: 0.0 km²
Final geometry valid: True

Processing: Mumbai
Features: 24
Invalid source features: 0
Union area: 458.08 km²
Overlap excess: -0.0 km²
Final geometry valid: True

Processing: Hyderabad
Features: 1
Invalid source features: 0
Union area: 608.87 km²
Overlap excess: 0.0 km²
Final geometry valid: True

Processing: Chennai
Features: 201
Invalid source features: 0
Union area: 434.6 km²
Overlap excess: 0.0 km²
Final geometry valid: False

Processing: Kolkata
Features: 141
Invalid source features: 1
Union area: 186.61 km²
Overlap excess: 0.02 km²
Final geometry valid: False

Processing: Ahmedabad
Features: 48
Invalid source features: 0
Union area: 438.44 km²
Overlap excess: 0.0 km²
Final geometry valid: True

Processing: Jaipur
Features: 1
Invalid source features: 0
Union area: 376.22 km²
Overlap excess: 0.0

,city,source,source_type,features,invalid_features_repaired,original_area_sum_km2,final_union_area_km2,repair_area_change_km2,overlap_excess_km2,final_geometry_valid,final_geometry_type,saved_path
0,Pune,Pune 2022 ward union,Ward union,1,0,507.1326,507.1326,0.0000,0.0000,True,Polygon,/kaggle/working/geo_data/boundaries/final/pune...
1,Mumbai,BMC_Wards.geojson,Ward union,24,0,458.0828,458.0828,0.0000,-0.0000,True,MultiPolygon,/kaggle/working/geo_data/boundaries/final/mumb...
2,Hyderabad,ghmc-area.geojson,Direct municipal boundary,1,0,608.8681,608.8681,-0.0000,0.0000,True,MultiPolygon,/kaggle/working/geo_data/boundaries/final/hyde...
3,Chennai,Wards.geojson,Ward union,201,0,434.5987,434.5987,-0.0000,0.0000,False,Polygon,/kaggle/working/geo_data/boundaries/final/chen...
4,Kolkata,kolkata.geojson,Ward union,141,1,186.6332,186.6131,-0.0201,0.0201,False,MultiPolygon,/kaggle/working/geo_data/boundaries/final/kolk...
5,Ahmedabad,Amdavad Wards Map 2024,Ward union,48,0,438.4398,438.4398,-0.0000,0.0000,True,Polygon,/kaggle/working/geo_data/boundaries/final/ahme...
6,Jaipur,Jaipur_Boundary.geojson,Direct municipal boundary,1,0,376.2189,376.2189,0.0000,0.0000,True,Polygon,/kaggle/working/geo_data/boundaries/final/jaip...
7,Lucknow,Lucknow_city_boundary.geojson,Direct municipal boundary,1,0,283.5793,283.5793,0.0000,0.0000,True,Polygon,/kaggle/working/geo_data/boundaries/final/luck...
8,Bhubaneswar,BMC Boundary.GeoJSON,Direct municipal boundary,1,0,146.8613,146.8613,0.0000,0.0000,True,Polygon,/kaggle/working/geo_data/boundaries/final/bhub...
9,Indore,Indore Wards Map 2024 — 85 wards,Ward union,85,0,268.8820,268.8820,-0.0000,0.0000,True,Polygon,/kaggle/working/geo_data/boundaries/final/indo...



Combined boundary file: /kaggle/working/geo_data/boundaries/final/final_10_city_boundaries.geojson
Validation manifest: /kaggle/working/geo_data/boundaries/final/boundary_validation_manifest.csv
Total cities: 10
All final geometries valid: False

BOUNDARY STAGE COMPLETE.


In [17]:

import pandas as pd
from pathlib import Path

BASE = Path("/kaggle/input/datasets/vedantsaraswate/udise-2526")

P1_PATH = BASE / "profile_data_1_All State_2025-26" / "100_prof1.csv"

prof = pd.read_csv(P1_PATH, low_memory=False)

# Find relevant administrative columns
admin_cols = [
    c for c in prof.columns
    if any(term in c.lower() for term in
           ["local", "body", "district", "block", "urban"])
]

print("Administrative columns:")
print(admin_cols)

# Search for local-body-like columns
local_cols = [
    c for c in prof.columns
    if "local" in c.lower() or "body" in c.lower()
]

cities = [
    "Pune", "Mumbai", "Hyderabad", "Chennai", "Kolkata",
    "Ahmedabad", "Jaipur", "Lucknow", "Bhubaneswar", "Indore"
]

for col in local_cols:
    print(f"\n\n--- {col} ---")

    values = (
        prof[col]
        .dropna()
        .astype(str)
        .drop_duplicates()
    )

    for city in cities:
        matches = values[
            values.str.contains(city, case=False, regex=False)
        ].tolist()

        if matches:
            print(f"{city}: {matches[:30]}")


Administrative columns:
['district', 'block', 'rural_urban', 'lgd_urban_local_body_name', 'lgd_block_name']


--- lgd_urban_local_body_name ---
Pune: ['Pune-Municipal Corporations', 'Pune-Cantonment Board', 'Singampuneri']
Mumbai: ['Greater Mumbai_Municipal Corporations', 'Navi Mumbai-Municipal Corporations', 'Greater Mumbai-Municipal Corporations']
Hyderabad: ['Hyderabad-Municipal Corporations', 'Hyderabad - Municipal Corporation']
Chennai: ['Chennai', 'Greater Chennai Corporation - Municipal Corporation', 'Greater Chennai Corporation']
Kolkata: ['Kolkata-Municipal Corporations']
Jaipur: ['Greater Jaipur-Municipal Corporations', 'Jaipur Heritage-Municipal Corporations', 'JAIJAIPUR-Town Panchayat']
Lucknow: ['Lucknow-Municipal Corporations', 'Amethi(lucknow)-Town Panchayat']
Bhubaneswar: ['Bhubaneswar-Municipal Corporations']
Indore: ['Indore-Municipal Corporations']


In [18]:

# Find Ahmedabad/Amdavad administrative labels
for col in ["lgd_urban_local_body_name", "district"]:
    print(f"\n--- {col} ---")

    vals = (
        prof[col]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .sort_values()
    )

    matches = vals[
        vals.str.contains(
            "ahmed|amdavad|amd",
            case=False,
            regex=True
        )
    ]

    print(matches.to_list())



--- lgd_urban_local_body_name ---
['Aamdi-Town Panchayat', 'Ahmedgarh-Municipality', 'Ahmednagar-Cantonment Board', 'Dhamdha-Town Panchayat', 'Mamdot-Town Panchayat', 'Municipal Committee Seer Hamdan', 'Ramdas', 'Ramdurg-Municipality', 'Samdhan-Town Panchayat', 'Wangjing Lamding Municipal Council_Municipality']

--- district ---
['AHMEDABAD']


In [19]:

# Inspect Ahmedabad UDISE records

ahm = prof[
    prof["district"].astype(str).str.strip().str.upper()
    == "AHMEDABAD"
].copy()

print("Total Ahmedabad district records:", len(ahm))

print("\nRural/urban classification:")
display(ahm["rural_urban"].value_counts(dropna=False))

print("\nLocal body values:")
display(
    ahm["lgd_urban_local_body_name"]
    .value_counts(dropna=False)
    .head(40)
)

print("\nBlock names:")
display(
    ahm["block"]
    .value_counts(dropna=False)
)

print("\nLGD block names:")
display(
    ahm["lgd_block_name"]
    .value_counts(dropna=False)
)

print("\nSample Ahmedabad records:")
display(
    ahm[
        [
            "district",
            "block",
            "rural_urban",
            "lgd_urban_local_body_name",
            "lgd_block_name"
        ]
    ].head(20)
)


Total Ahmedabad district records: 3941

Rural/urban classification:


rural_urban
2    2888
1    1053
Name: count, dtype: int64


Local body values:


lgd_urban_local_body_name
Ahmadabad-Municipal Corporations    2625
NaN                                 1059
Dholka-Municipality                   71
Bopal-ghuma-Municipality              43
Viramgam-Municipality                 40
Sanand-Municipality                   37
Dhandhuka-Municipality                30
Bavla-Municipality                    22
Bareja-Municipality                   14
Name: count, dtype: int64


Block names:


block
AMC                 2438
DASCROI              367
DHOLKA               207
SANAND               200
CITY                 187
VIRAMGAM             144
BAVLA                109
DHANDHUKA             89
DETROJ-RAMPURA        71
MANDAL                69
DHOLERA               54
AHMEDABAD REGION       5
PUNE REGION            1
Name: count, dtype: int64


LGD block names:


lgd_block_name
NaN               2892
Daskroi            293
Sanand             179
Dholka             135
Viramgam           105
Bavla               85
Detroj Rampura      70
Mandal              69
Dhandhuka           59
Dholera             54
Name: count, dtype: int64


Sample Ahmedabad records:


,district,block,rural_urban,lgd_urban_local_body_name,lgd_block_name
743,AHMEDABAD,AMC,2,Ahmadabad-Municipal Corporations,NaN
766,AHMEDABAD,AMC,2,Ahmadabad-Municipal Corporations,NaN
831,AHMEDABAD,AMC,2,Ahmadabad-Municipal Corporations,NaN
851,AHMEDABAD,AMC,2,Ahmadabad-Municipal Corporations,NaN
960,AHMEDABAD,VIRAMGAM,1,NaN,Viramgam
977,AHMEDABAD,CITY,2,Ahmadabad-Municipal Corporations,NaN
1006,AHMEDABAD,CITY,2,Ahmadabad-Municipal Corporations,NaN
1056,AHMEDABAD,AMC,2,Ahmadabad-Municipal Corporations,NaN
1200,AHMEDABAD,AMC,2,Ahmadabad-Municipal Corporations,NaN
1962,AHMEDABAD,DASCROI,1,NaN,Daskroi


In [22]:

# ============================================================
# HiMCM — CITY EDUCATION INDEX
# STAGE 1: SEVEN-FACTOR SCHOOL QUALITY
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("/kaggle/input/datasets/vedantsaraswate/udise-2526")
OUT = Path("/kaggle/working/geo_data/outputs")
OUT.mkdir(parents=True, exist_ok=True)

P1_PATH = BASE / "profile_data_1_All State_2025-26" / "100_prof1.csv"
FAC_PATH = BASE / "facility_data_All State_2025-26" / "100_fac.csv"
TCH_PATH = BASE / "teacher_data_All State_2025-26" / "100_tch.csv"

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

prof = pd.read_csv(P1_PATH, low_memory=False)
fac = pd.read_csv(FAC_PATH, low_memory=False)
tch = pd.read_csv(TCH_PATH, low_memory=False)

# Select only columns needed from each dataset
p1_cols = [
    "pseudocode", "district", "block", "rural_urban",
    "lgd_urban_local_body_name", "lowclass", "highclass"
]

fac_cols = [
    "pseudocode",
    "classrooms_in_good_condition", "total_class_rooms",
    "total_boys_func_toilet", "total_girls_func_toilet",
    "total_boys_toilet", "total_girls_toilet",
    "electricity_availability",
    "hand_pump_fun_yn", "well_prot_fun_yn",
    "tap_fun_yn", "othsrc_fun_yn", "pack_water_fun_yn",
    "library_availability",
    "phy_lab_cond", "chem_lab_cond", "bio_lab_cond",
    "comp_lab_cond",
    "math_lab_cond", "geo_lab_cond", "lang_lab_cond",
    "home_sc_lab_cond", "psycho_lab_cond"
]

tch_cols = [
    "pseudocode",
    "total_tch", "graduate", "post_graduate_and_above",
    "diploma_certificate", "diploma_ele_edu",
    "bachelor_of_ee", "bed_equivalent",
    "diploma_special_edu", "med_equivalent", "none"
]

for df, cols, name in [
    (prof, p1_cols, "Profile"),
    (fac, fac_cols, "Facility"),
    (tch, tch_cols, "Teacher")
]:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"{name} dataset missing columns: {missing}")

prof = prof[p1_cols].copy()
fac = fac[fac_cols].copy()
tch = tch[tch_cols].copy()

# Ensure unique join keys
for df, name in [(prof, "Profile"), (fac, "Facility"), (tch, "Teacher")]:
    if df["pseudocode"].duplicated().any():
        raise ValueError(f"Duplicate pseudocode in {name} dataset.")

df = prof.merge(fac, on="pseudocode", how="inner")
df = df.merge(tch, on="pseudocode", how="inner")

print("Joined schools:", len(df))

# ------------------------------------------------------------
# 2. NUMERIC CONVERSION
# ------------------------------------------------------------

numeric_cols = [
    c for c in df.columns
    if c not in [
        "pseudocode", "district", "block",
        "rural_urban", "lgd_urban_local_body_name"
    ]
]

for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# ------------------------------------------------------------
# 3. SECONDARY SCHOOL ELIGIBILITY
# ------------------------------------------------------------

df = df[
    (df["lowclass"] <= 12) &
    (df["highclass"] >= 9)
].copy()

print("Eligible secondary schools:", len(df))

# ------------------------------------------------------------
# 4. SCHOOL QUALITY FACTORS
# ------------------------------------------------------------

# Factor 1: Classroom condition
df["C"] = (
    df["classrooms_in_good_condition"] /
    df["total_class_rooms"].replace(0, np.nan)
)

# Factor 2: Functional sanitation
toilets_total = (
    df["total_boys_toilet"] +
    df["total_girls_toilet"]
)

toilets_functional = (
    df["total_boys_func_toilet"] +
    df["total_girls_func_toilet"]
)

df["S"] = (
    toilets_functional /
    toilets_total.replace(0, np.nan)
)

# Factor 3: Electricity
df["E"] = np.where(
    df["electricity_availability"] == 1, 1.0,
    np.where(
        df["electricity_availability"].isin([2, 3]),
        0.0, np.nan
    )
)

# Factor 4: Functional water source
water_cols = [
    "hand_pump_fun_yn", "well_prot_fun_yn",
    "tap_fun_yn", "othsrc_fun_yn", "pack_water_fun_yn"
]

water = df[water_cols]

df["W"] = np.where(
    water.isin([1]).any(axis=1), 1.0,
    np.where(
        water.notna().all(axis=1),
        0.0, np.nan
    )
)

# Factor 5: Academic teacher qualification
df["AQ"] = (
    df["graduate"] + df["post_graduate_and_above"]
) / df["total_tch"].replace(0, np.nan)

# Factor 6: Learning resources
df["Lib_A"] = np.where(
    df["library_availability"] == 1, 1.0,
    np.where(df["library_availability"] == 2, 0.0, np.nan)
)

lab_cols = [
    "phy_lab_cond", "chem_lab_cond",
    "bio_lab_cond", "comp_lab_cond"
]

for c in lab_cols:
    df[c + "_score"] = np.where(
        df[c].isin([1, 2, 3]), 1.0,
        np.where(df[c].isin([0, 4]), 0.0, np.nan)
    )

other_lab_cols = [
    "math_lab_cond", "geo_lab_cond",
    "lang_lab_cond", "home_sc_lab_cond",
    "psycho_lab_cond"
]

df["OtherLab"] = np.where(
    df[other_lab_cols].isin([1, 2, 3]).any(axis=1),
    1.0,
    np.where(
        df[other_lab_cols].notna().all(axis=1),
        0.0, np.nan
    )
)

lr_cols = [
    "Lib_A",
    "phy_lab_cond_score",
    "chem_lab_cond_score",
    "bio_lab_cond_score",
    "comp_lab_cond_score",
    "OtherLab"
]

df["LR"] = df[lr_cols].mean(axis=1)

# Factor 7: Teacher professional qualification
D = (
    df["diploma_certificate"].fillna(0) +
    df["diploma_ele_edu"].fillna(0)
)

B = (
    df["bachelor_of_ee"].fillna(0) +
    df["bed_equivalent"].fillna(0) +
    df["diploma_special_edu"].fillna(0)
)

M = df["med_equivalent"].fillna(0)
N = df["none"].fillna(0)

pt_denominator = 3 * (D + B + M + N)

df["PT"] = np.where(
    pt_denominator > 0,
    (D + 2 * B + 3 * M) / pt_denominator,
    np.nan
)

# ------------------------------------------------------------
# 5. SEVEN-FACTOR SCHOOL QUALITY SCORE
# ------------------------------------------------------------

quality_cols = ["C", "S", "E", "W", "AQ", "LR", "PT"]

# Require all seven factors for a complete school score
df["Q7"] = df[quality_cols].mean(axis=1)

df["Q7_complete"] = df[quality_cols].notna().all(axis=1)

print(
    "Complete seven-factor records:",
    int(df["Q7_complete"].sum())
)

print(
    "Incomplete seven-factor records:",
    int((~df["Q7_complete"]).sum())
)

# ------------------------------------------------------------
# 6. CITY-LEVEL LOCAL BODY MATCHING
# ------------------------------------------------------------

CITY_LABELS = {
    "Pune": [
        "Pune-Municipal Corporations"
    ],

    "Mumbai": [
        "Greater Mumbai_Municipal Corporations",
        "Greater Mumbai-Municipal Corporations"
    ],

    "Hyderabad": [
        "Hyderabad-Municipal Corporations",
        "Hyderabad - Municipal Corporation"
    ],

    "Chennai": [
    "Chennai",
    "Greater Chennai Corporation - Municipal Corporation",
    "Greater Chennai Corporation"
],

    "Kolkata": [
        "Kolkata-Municipal Corporations"
    ],

    "Ahmedabad": [
        "Ahmadabad-Municipal Corporations"
    ],

    "Jaipur": [
        "Greater Jaipur-Municipal Corporations",
        "Jaipur Heritage-Municipal Corporations"
    ],

    "Lucknow": [
        "Lucknow-Municipal Corporations"
    ],

    "Bhubaneswar": [
        "Bhubaneswar-Municipal Corporations"
    ],

    "Indore": [
        "Indore-Municipal Corporations"
    ]
}

city_results = []
city_school_frames = []

for city, labels in CITY_LABELS.items():

    city_df = df[
        df["lgd_urban_local_body_name"].isin(labels)
    ].copy()

    complete = city_df[city_df["Q7_complete"]]

    city_results.append({
        "city": city,
        "matched_secondary_schools": len(city_df),
        "complete_quality_schools": len(complete),
        "incomplete_quality_schools": len(city_df) - len(complete),
        "completeness_percent": (
            100 * len(complete) / len(city_df)
            if len(city_df) else np.nan
        ),
        "mean_C": city_df["C"].mean(),
        "mean_S": city_df["S"].mean(),
        "mean_E": city_df["E"].mean(),
        "mean_W": city_df["W"].mean(),
        "mean_AQ": city_df["AQ"].mean(),
        "mean_LR": city_df["LR"].mean(),
        "mean_PT": city_df["PT"].mean(),
        "mean_Q7": complete["Q7"].mean(),
        "std_Q7": complete["Q7"].std()
    })

    city_df["city"] = city
    city_school_frames.append(city_df)

    print(
        f"{city}: {len(city_df)} matched, "
        f"{len(complete)} complete"
    )

city_quality = pd.DataFrame(city_results)

city_quality_path = OUT / "city_school_quality_10.csv"
city_quality.to_csv(city_quality_path, index=False)

school_quality = pd.concat(
    city_school_frames,
    ignore_index=True
)

school_quality_path = OUT / "school_quality_10_cities.csv"
school_quality.to_csv(school_quality_path, index=False)

print("\nCITY-LEVEL QUALITY RESULTS")
display(
    city_quality[
        [
            "city",
            "matched_secondary_schools",
            "complete_quality_schools",
            "completeness_percent",
            "mean_Q7",
            "std_Q7"
        ]
    ].round(4)
)

print("\nSaved:", city_quality_path)
print("Saved:", school_quality_path)


Joined schools: 1466682
Eligible secondary schools: 311700
Complete seven-factor records: 307599
Incomplete seven-factor records: 4101
Pune: 750 matched, 748 complete
Mumbai: 2035 matched, 2031 complete
Hyderabad: 3975 matched, 3956 complete
Chennai: 1114 matched, 1111 complete
Kolkata: 726 matched, 694 complete
Ahmedabad: 941 matched, 924 complete
Jaipur: 1551 matched, 1466 complete
Lucknow: 764 matched, 758 complete
Bhubaneswar: 184 matched, 180 complete
Indore: 584 matched, 569 complete

CITY-LEVEL QUALITY RESULTS


,city,matched_secondary_schools,complete_quality_schools,completeness_percent,mean_Q7,std_Q7
0,Pune,750,748,99.7333,0.8699,0.0619
1,Mumbai,2035,2031,99.8034,0.8389,0.0567
2,Hyderabad,3975,3956,99.5220,0.8186,0.0562
3,Chennai,1114,1111,99.7307,0.8994,0.0587
4,Kolkata,726,694,95.5923,0.8519,0.0785
5,Ahmedabad,941,924,98.1934,0.8776,0.0594
6,Jaipur,1551,1466,94.5197,0.8567,0.0817
7,Lucknow,764,758,99.2147,0.8686,0.0750
8,Bhubaneswar,184,180,97.8261,0.8561,0.0718
9,Indore,584,569,97.4315,0.8664,0.0764



Saved: /kaggle/working/geo_data/outputs/city_school_quality_10.csv
Saved: /kaggle/working/geo_data/outputs/school_quality_10_cities.csv


In [21]:

# Diagnose Chennai local-body records

chennai_labels = (
    prof["lgd_urban_local_body_name"]
    .dropna()
    .astype(str)
)

print(
    chennai_labels[
        chennai_labels.str.contains(
            "Chennai", case=False, regex=False
        )
    ].value_counts()
)


lgd_urban_local_body_name
Chennai                                                2050
Greater Chennai Corporation - Municipal Corporation      68
Greater Chennai Corporation                               3
Name: count, dtype: int64


In [24]:
import shutil
from IPython.display import FileLink, display

output_dir = "/kaggle/working/geo_data/outputs"

zip_path = shutil.make_archive(
    "/kaggle/working/HiMCM_quality_outputs",
    "zip",
    output_dir
)

print("ZIP created:", zip_path)
display(FileLink(zip_path))

ZIP created: /kaggle/working/HiMCM_quality_outputs.zip


/kaggle/working/HiMCM_quality_outputs.zip

In [29]:

# ============================================================
# HiMCM — OSM SCHOOL ACQUISITION
# FINAL CHECKPOINTED VERSION
# ============================================================

import os
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import shapely

from pathlib import Path

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

BASE_DIR = Path("/kaggle/working/geo_data")

BOUNDARY_PATH = (
    BASE_DIR / "boundaries/final/final_10_city_boundaries.geojson"
)

SCHOOL_DIR = BASE_DIR / "schools_raw"
SCHOOL_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = SCHOOL_DIR / "osm_acquisition_log.csv"
COMBINED_PATH = SCHOOL_DIR / "osm_schools_all_cities.geojson"

OVERPASS_URL = "https://overpass-api.de/api"

ox.settings.use_cache = True
ox.settings.requests_timeout = 180
ox.settings.overpass_rate_limit = True
ox.settings.overpass_url = OVERPASS_URL

# ------------------------------------------------------------
# 2. LOAD BOUNDARIES
# ------------------------------------------------------------

boundaries = gpd.read_file(BOUNDARY_PATH).to_crs("EPSG:4326")

if len(boundaries) != 10:
    raise ValueError("Expected exactly 10 city boundaries.")

# Repair invalid boundaries if necessary
boundaries["geometry"] = shapely.make_valid(
    boundaries.geometry.array
)

print("Loaded cities:", len(boundaries))


# ------------------------------------------------------------
# 3. HELPER FUNCTION
# ------------------------------------------------------------

def get_tag(result, tag):
    if tag in result.columns:
        return result[tag].values
    return [None] * len(result)


# ------------------------------------------------------------
# 4. CITY-BY-CITY ACQUISITION
# ------------------------------------------------------------

successful_files = []
logs = []

for _, row in boundaries.iterrows():

    city = row["city"]
    polygon = row.geometry

    output_path = SCHOOL_DIR / f"{city.lower()}_osm_schools.geojson"

    print("\n" + "=" * 55)
    print("CITY:", city)
    print("=" * 55)

    # Reuse a previous successful acquisition
    if output_path.exists():

        try:
            existing = gpd.read_file(output_path)

            if len(existing) > 0:
                print("Using saved checkpoint:", len(existing))

                successful_files.append(output_path)

                logs.append({
                    "city": city,
                    "status": "Loaded checkpoint",
                    "schools": len(existing)
                })

                continue

        except Exception:
            print("Existing checkpoint unreadable; reacquiring.")

    try:

        # One Overpass request per city
        result = ox.features_from_polygon(
            polygon,
            tags={"amenity": "school"}
        )

        print("OSM features returned:", len(result))

        if result.empty:
            logs.append({
                "city": city,
                "status": "No features returned",
                "schools": 0
            })
            continue

        result = result.to_crs("EPSG:4326")

        # Correct conversion of OSM MultiIndex
        osm_ids = [
            f"{idx[0]}/{idx[1]}"
            for idx in result.index.to_list()
        ]

        # Representative point for each school geometry
        points = result.geometry.representative_point()

        # Construct a clean dataset
        schools = gpd.GeoDataFrame(
            {
                "osm_id": osm_ids,
                "name": get_tag(result, "name"),
                "amenity": get_tag(result, "amenity"),
                "operator": get_tag(result, "operator"),
                "operator_type": get_tag(result, "operator:type"),
                "grades": get_tag(result, "grades"),
                "min_age": get_tag(result, "min_age"),
                "max_age": get_tag(result, "max_age"),
                "isced_level": get_tag(result, "isced:level"),
                "education": get_tag(result, "education"),
                "school_type": get_tag(result, "school:type"),
                "original_geometry_type": result.geometry.geom_type.values,
                "geometry": points.values
            },
            geometry="geometry",
            crs="EPSG:4326"
        )

        # Retain only points inside the municipal boundary
        schools = schools.loc[
            schools.geometry.apply(polygon.covers)
        ].copy()

        schools["city"] = city

        # Save immediately
        schools.to_file(
            output_path,
            driver="GeoJSON"
        )

        successful_files.append(output_path)

        print("Schools inside boundary:", len(schools))
        print("Saved:", output_path)

        logs.append({
            "city": city,
            "status": "Success",
            "schools": len(schools)
        })

    except Exception as e:

        print("FAILED:", city)
        print(type(e).__name__, str(e)[:400])

        logs.append({
            "city": city,
            "status": "Failed",
            "schools": 0,
            "error": str(e)[:400]
        })

    # Persist log after every city
    pd.DataFrame(logs).to_csv(LOG_PATH, index=False)

    # Small pause between requests
    time.sleep(2)


# ------------------------------------------------------------
# 5. COMBINE CITY FILES
# ------------------------------------------------------------

if not successful_files:
    raise RuntimeError(
        "No school files were successfully acquired."
    )

frames = [
    gpd.read_file(path)
    for path in successful_files
]

combined = gpd.GeoDataFrame(
    pd.concat(frames, ignore_index=True),
    geometry="geometry",
    crs="EPSG:4326"
)

# Remove duplicate OSM elements
combined = combined.drop_duplicates(
    subset=["osm_id", "city"]
)

combined.to_file(
    COMBINED_PATH,
    driver="GeoJSON"
)


# ------------------------------------------------------------
# 6. ELIGIBILITY DATA COVERAGE
# ------------------------------------------------------------

eligibility_cols = [
    "grades",
    "min_age",
    "max_age",
    "isced_level"
]

combined["has_eligibility_data"] = (
    combined[eligibility_cols].notna().any(axis=1)
)

coverage = (
    combined.groupby("city")
    .agg(
        total_schools=("osm_id", "count"),
        schools_with_eligibility_tags=(
            "has_eligibility_data", "sum"
        )
    )
    .reset_index()
)

coverage["tag_coverage_percent"] = (
    100 * coverage["schools_with_eligibility_tags"]
    / coverage["total_schools"]
)

coverage.to_csv(
    SCHOOL_DIR / "osm_eligibility_tag_coverage.csv",
    index=False
)


# ------------------------------------------------------------
# 7. FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("OSM ACQUISITION FINISHED")
print("=" * 60)

display(pd.DataFrame(logs))
display(coverage.round(2))

print("Total school features:", len(combined))
print("Combined file:", COMBINED_PATH)
print("Acquisition log:", LOG_PATH)


Loaded cities: 10

CITY: Pune
FAILED: Pune
ResponseStatusCodeError 'overpass-api.de' responded: 502 Bad Gateway # 502 Gateway Timeout

This Overpass API server is currently at capacity. All query workers are busy
and the request queue is full.

## What to do

Retry your request using exponential backoff: wait a short interval, then retry.
If the request fails again, double the wait time and retry again.

## For heavy or automated use

This public endpoint is inte

CITY: Mumbai
OSM features returned: 546
Schools inside boundary: 546
Saved: /kaggle/working/geo_data/schools_raw/mumbai_osm_schools.geojson

CITY: Hyderabad
OSM features returned: 466
Schools inside boundary: 465
Saved: /kaggle/working/geo_data/schools_raw/hyderabad_osm_schools.geojson

CITY: Chennai
FAILED: Chennai
TypeError Boundaries must be a Polygon or MultiPolygon. If you requested `features_from_place`, ensure your query geocodes to a Polygon or MultiPolygon. See the documentation for details.

CITY: Kolkata
OSM featur

,city,status,schools,error
0,Pune,Failed,0,'overpass-api.de' responded: 502 Bad Gateway #...
1,Mumbai,Success,546,NaN
2,Hyderabad,Success,465,NaN
3,Chennai,Failed,0,Boundaries must be a Polygon or MultiPolygon. ...
4,Kolkata,Success,119,NaN
5,Ahmedabad,Success,69,NaN
6,Jaipur,Failed,0,'overpass-api.de' responded: 502 Bad Gateway #...
7,Lucknow,Failed,0,'overpass-api.de' responded: 502 Bad Gateway #...
8,Bhubaneswar,Success,33,NaN
9,Indore,Success,254,NaN


,city,total_schools,schools_with_eligibility_tags,tag_coverage_percent
0,Ahmedabad,69,7,10.14
1,Bhubaneswar,33,4,12.12
2,Hyderabad,465,18,3.87
3,Indore,254,164,64.57
4,Kolkata,119,6,5.04
5,Mumbai,546,43,7.88


Total school features: 1486
Combined file: /kaggle/working/geo_data/schools_raw/osm_schools_all_cities.geojson
Acquisition log: /kaggle/working/geo_data/schools_raw/osm_acquisition_log.csv


In [30]:
import os
import pandas as pd

ROOT = "/kaggle/working/geo_data"

print("ROOT EXISTS:", os.path.exists(ROOT))

for folder, dirs, files in os.walk(ROOT):
    level = folder.replace(ROOT, "").count(os.sep)
    if level > 3:
        dirs[:] = []
        continue

    print(f"\n{'  ' * level}{os.path.basename(folder)}/")

    for name in sorted(files):
        path = os.path.join(folder, name)
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{'  ' * (level + 1)}{name} — {size_mb:.2f} MB")

print("\n--- OSM ACQUISITION LOG ---")
log_path = os.path.join(ROOT, "schools_raw", "osm_acquisition_log.csv")

if os.path.exists(log_path):
    display(pd.read_csv(log_path))
else:
    print("No acquisition log found.")

ROOT EXISTS: True

geo_data/

  schools_raw/
    ahmedabad_osm_schools.geojson — 0.03 MB
    bhubaneswar_osm_schools.geojson — 0.01 MB
    hyderabad_osm_schools.geojson — 0.18 MB
    indore_osm_schools.geojson — 0.10 MB
    kolkata_osm_schools.geojson — 0.05 MB
    mumbai_osm_schools.geojson — 0.22 MB
    osm_acquisition_log.csv — 0.00 MB
    osm_eligibility_tag_coverage.csv — 0.00 MB
    osm_schools_all_cities.geojson — 0.59 MB

  population/

  logs/
    boundary_source_validation.csv — 0.00 MB
    geometry_diagnostics.csv — 0.00 MB

  schools_processed/

  outputs/
    city_school_quality_10.csv — 0.00 MB
    school_quality_10_cities.csv — 3.08 MB

  boundaries/
    ahmedabad_wards_2024_candidate.geojson — 0.33 MB
    pune_boundary.geojson — 0.03 MB
    pune_wards_2022.geojson — 0.24 MB

    raw_sources/
      ahmedabad_source.geojson — 1.18 MB
      ahmedabad_wards_2024.kml — 0.28 MB
      bengaluru_source.geojson — 2.01 MB
      bhubaneswar_source.geojson — 0.38 MB
      chennai_s

,city,status,schools,error
0,Pune,Failed,0,'overpass-api.de' responded: 502 Bad Gateway #...
1,Mumbai,Success,546,NaN
2,Hyderabad,Success,465,NaN
3,Chennai,Failed,0,Boundaries must be a Polygon or MultiPolygon. ...
4,Kolkata,Success,119,NaN
5,Ahmedabad,Success,69,NaN
6,Jaipur,Failed,0,'overpass-api.de' responded: 502 Bad Gateway #...
7,Lucknow,Failed,0,'overpass-api.de' responded: 502 Bad Gateway #...
8,Bhubaneswar,Success,33,NaN
9,Indore,Success,254,NaN


In [31]:
import os
import time
import geopandas as gpd
import pandas as pd
import osmnx as ox

from shapely import make_valid, union_all
from shapely.geometry import Polygon, MultiPolygon, GeometryCollection

ROOT = "/kaggle/working/geo_data"
BOUNDARY_DIR = ROOT + "/boundaries/final"
SCHOOL_DIR = ROOT + "/schools_raw"

FAILED_CITIES = ["Pune", "Chennai", "Jaipur", "Lucknow"]

ENDPOINTS = [
    "https://overpass.kumi.systems/api",
    "https://overpass-api.de/api",
    "https://overpass.nchc.org.tw/api"
]

ox.settings.use_cache = True
ox.settings.requests_timeout = 180

def polygon_parts(geom):
    if geom is None or geom.is_empty:
        return []
    if isinstance(geom, Polygon):
        return [geom]
    if isinstance(geom, MultiPolygon):
        return list(geom.geoms)
    if isinstance(geom, GeometryCollection):
        parts = []
        for item in geom.geoms:
            parts.extend(polygon_parts(item))
        return parts
    return []

def load_city_polygon(city):
    path = f"{BOUNDARY_DIR}/{city.lower()}_boundary.geojson"
    boundary = gpd.read_file(path).to_crs("EPSG:4326")

    geom = make_valid(union_all(boundary.geometry))
    parts = polygon_parts(geom)

    if not parts:
        raise ValueError(f"No polygon geometry found for {city}")

    return make_valid(union_all(parts))

results = []

for city in FAILED_CITIES:
    output_path = f"{SCHOOL_DIR}/{city.lower()}_osm_schools.geojson"

    print(f"\n{'='*55}\n{city}")

    try:
        polygon = load_city_polygon(city)
        success = False

        for endpoint in ENDPOINTS:
            print(f"Trying: {endpoint}")
            ox.settings.overpass_url = endpoint

            try:
                raw = ox.features_from_polygon(
                    polygon,
                    tags={"amenity": "school"}
                )

                if raw.empty:
                    print("No features returned.")
                    continue

                raw = raw.copy()
                raw["osm_id"] = [
                    f"{idx[0]}/{idx[1]}"
                    for idx in raw.index.to_list()
                ]
                raw = raw.reset_index(drop=True)
                raw["city"] = city

                raw = gpd.GeoDataFrame(
                    raw, geometry="geometry", crs="EPSG:4326"
                )

                raw.to_file(output_path, driver="GeoJSON")

                print(f"SUCCESS: saved {len(raw)} schools.")
                results.append((city, "Success", len(raw), ""))
                success = True
                break

            except Exception as e:
                print(f"Endpoint failed: {str(e)[:250]}")
                time.sleep(10)

        if not success:
            results.append((city, "Failed", 0, "All endpoints failed"))

    except Exception as e:
        print(f"CITY FAILED: {str(e)[:300]}")
        results.append((city, "Failed", 0, str(e)[:500]))

# Rebuild combined file from every available per-city file
frames = []

for city in [
    "Pune", "Mumbai", "Hyderabad", "Chennai", "Kolkata",
    "Ahmedabad", "Jaipur", "Lucknow", "Bhubaneswar", "Indore"
]:
    path = f"{SCHOOL_DIR}/{city.lower()}_osm_schools.geojson"

    if os.path.exists(path):
        gdf = gpd.read_file(path)
        gdf["city"] = city
        frames.append(gdf)

if frames:
    combined = gpd.GeoDataFrame(
        pd.concat(frames, ignore_index=True),
        geometry="geometry",
        crs="EPSG:4326"
    )

    if "osm_id" in combined.columns:
        combined = combined.drop_duplicates(
            subset=["city", "osm_id"]
        )

    combined.to_file(
        f"{SCHOOL_DIR}/osm_schools_all_cities.geojson",
        driver="GeoJSON"
    )

    print("\nCOMBINED SCHOOL DATA")
    print("Total records:", len(combined))
    print(combined.groupby("city").size().to_string())

print("\nRETRY SUMMARY")
print(pd.DataFrame(
    results,
    columns=["city", "status", "schools", "error"]
).to_string(index=False))


Pune
Trying: https://overpass.kumi.systems/api
Endpoint failed: HTTPSConnectionPool(host='overpass.kumi.systems', port=443): Read timed out. (read timeout=180)
Trying: https://overpass-api.de/api
Endpoint failed: 'overpass-api.de' responded: 502 Bad Gateway # 502 Gateway Timeout

This Overpass API server is currently at capacity. All query workers are busy
and the request queue is full.

## What to do

Retry your request using exponential backoff: wait a shor
Trying: https://overpass.nchc.org.tw/api


KeyboardInterrupt: 

In [32]:
import shutil

print("Disk space:")
print(shutil.disk_usage("/kaggle/working"))

print("\nRequired regional downloads: approximately 1.3 GB")

Disk space:
usage(total=20957446144, used=19427328, free=20921241600)

Required regional downloads: approximately 1.3 GB


In [33]:
import os
import requests

PBF_DIR = "/kaggle/working/geo_data/osm_pbf"
os.makedirs(PBF_DIR, exist_ok=True)

PBF_PATH = PBF_DIR + "/india-latest.osm.pbf"
URL = "https://download.geofabrik.de/asia/india-latest.osm.pbf"

if not os.path.exists(PBF_PATH):
    print("Starting India OSM download...")

    with requests.get(URL, stream=True, timeout=120) as r:
        r.raise_for_status()

        total = int(r.headers.get("content-length", 0))
        downloaded = 0

        with open(PBF_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)

                    if total:
                        print(
                            f"\rDownloaded: {downloaded/1e9:.2f} / "
                            f"{total/1e9:.2f} GB",
                            end=""
                        )

    print("\nDownload complete.")
else:
    print("India PBF already exists. Skipping download.")

print("File size:", round(os.path.getsize(PBF_PATH)/1e9, 2), "GB")

Starting India OSM download...
Downloaded: 1.71 / 1.71 GB
Download complete.
File size: 1.71 GB


In [34]:
%pip install -q pyrosm

import os
import geopandas as gpd
import inspect
from pyrosm import OSM

ROOT = "/kaggle/working/geo_data"
PBF_PATH = ROOT + "/osm_pbf/india-latest.osm.pbf"
BOUNDARY_DIR = ROOT + "/boundaries/final"

print("PBF exists:", os.path.exists(PBF_PATH))
print("PBF size:", round(os.path.getsize(PBF_PATH)/1e9, 2), "GB")

print("\nPyrosm version:", __import__("pyrosm").__version__)
print("POI method:", inspect.signature(OSM.get_pois))

boundary_file = BOUNDARY_DIR + "/final_10_city_boundaries.geojson"
boundaries = gpd.read_file(boundary_file)

print("\nBoundary CRS:", boundaries.crs)
print("Boundary records:", len(boundaries))
print("Boundary columns:", boundaries.columns.tolist())
print("\nBoundary preview:")
display(boundaries.head())

Note: you may need to restart the kernel to use updated packages.
PBF exists: True
PBF size: 1.71 GB

Pyrosm version: 0.29.0
POI method: (self, custom_filter=None, extra_attributes=None, timestamp=None, tags_to_keep=None)

Boundary CRS: EPSG:4326
Boundary records: 10
Boundary columns: ['city', 'source', 'source_type', 'source_features', 'invalid_source_features', 'area_km2', 'geometry']

Boundary preview:


,city,source,source_type,source_features,invalid_source_features,area_km2,geometry
0,Pune,Pune 2022 ward union,Ward union,1,0,507.1326,"POLYGON ((73.83837 18.39517, 73.83692 18.3945,..."
1,Mumbai,BMC_Wards.geojson,Ward union,24,0,458.0828,"MULTIPOLYGON (((72.84575 18.94395, 72.84575 18..."
2,Hyderabad,ghmc-area.geojson,Direct municipal boundary,1,0,608.8681,"MULTIPOLYGON (((78.51267 17.33035, 78.51559 17..."
3,Chennai,Wards.geojson,Ward union,201,0,434.5987,"POLYGON ((80.19121 12.95732, 80.18986 12.95745..."
4,Kolkata,kolkata.geojson,Ward union,141,1,186.6131,"MULTIPOLYGON (((88.32544 22.45924, 88.32543 22..."


In [35]:

import os
import gc
import pandas as pd
import geopandas as gpd
from pyrosm import OSM

ROOT = "/kaggle/working/geo_data"
PBF_PATH = ROOT + "/osm_pbf/india-latest.osm.pbf"
BOUNDARY_PATH = ROOT + "/boundaries/final/final_10_city_boundaries.geojson"

OUT_DIR = ROOT + "/schools_raw/pbf_extracted"
os.makedirs(OUT_DIR, exist_ok=True)

boundaries = gpd.read_file(BOUNDARY_PATH).to_crs("EPSG:4326")

cities = [
    "Pune", "Mumbai", "Hyderabad", "Chennai", "Kolkata",
    "Ahmedabad", "Jaipur", "Lucknow", "Bhubaneswar", "Indore"
]

def get_school_features(osm, tag):
    try:
        return osm.get_pois(custom_filter={tag: ["school"]})
    except Exception as e:
        print(f"  {tag} extraction issue: {str(e)[:200]}")
        return None

summary = []

for city in cities:
    out_path = f"{OUT_DIR}/{city.lower()}_schools.geojson"

    if os.path.exists(out_path):
        existing = gpd.read_file(out_path)
        print(f"{city}: already extracted ({len(existing)} records), skipping.")
        summary.append((city, len(existing), "Existing checkpoint"))
        continue

    print(f"\n{'='*50}\nProcessing {city}")

    try:
        boundary = boundaries.loc[
            boundaries["city"] == city, "geometry"
        ].iloc[0]

        # Read only the city's bounding box from the PBF
        osm = OSM(PBF_PATH, bounding_box=boundary)

        frames = []

        for tag in ["amenity", "education"]:
            print(f"Extracting {tag}=school...")
            data = get_school_features(osm, tag)

            if data is not None and not data.empty:
                frames.append(data)

        if not frames:
            print("No school features found.")
            summary.append((city, 0, "No features"))
            continue

        schools = gpd.GeoDataFrame(
            pd.concat(frames, ignore_index=True),
            geometry="geometry",
            crs="EPSG:4326"
        )

        # Keep features intersecting the municipal boundary
        schools = schools[
            schools.geometry.notna() &
            schools.geometry.intersects(boundary)
        ].copy()

        # Deduplicate repeated records from the two tag queries
        id_cols = [
            c for c in ["osm_type", "id"]
            if c in schools.columns
        ]

        if len(id_cols) == 2:
            schools = schools.drop_duplicates(subset=id_cols)
        elif "id" in schools.columns:
            schools = schools.drop_duplicates(subset=["id"])

        schools["city"] = city

        schools.to_file(out_path, driver="GeoJSON")

        print(f"Saved {len(schools)} school features.")
        summary.append((city, len(schools), "Success"))

        del osm, schools, frames
        gc.collect()

    except Exception as e:
        print(f"FAILED {city}: {str(e)[:400]}")
        summary.append((city, 0, f"Failed: {str(e)[:200]}"))
        gc.collect()

print("\n========== EXTRACTION SUMMARY ==========")
summary_df = pd.DataFrame(
    summary, columns=["city", "schools", "status"]
)
display(summary_df)

summary_df.to_csv(
    ROOT + "/schools_raw/pbf_extraction_summary.csv",
    index=False
)



Processing Pune
Extracting amenity=school...


Reading india-latest.osm.pbf: indexing the file:   0%|          | 00:02<33:33

Extracting education=school...


Reading india-latest.osm.pbf: decoding the blocks:   2%|2         | 00:02<01:29

Saved 256 school features.

Processing Mumbai
Extracting amenity=school...


Reading india-latest.osm.pbf: decoding the blocks:   2%|2         | 00:02<01:33

... (34 outputs left out)

Reading india-latest.osm.pbf: decoding the blocks:   2%|2         | 00:02<01:29

/usr/local/lib/python3.13/dist-packages/pyrosm/utils/__init__.py:131: UserWarning: Could not find any POIs for given area.
  result = method(*args, **kwargs)


Saved 254 school features.

========== EXTRACTION SUMMARY ==========


,city,schools,status
0,Pune,256,Success
1,Mumbai,548,Success
2,Hyderabad,466,Success
3,Chennai,573,Success
4,Kolkata,119,Success
5,Ahmedabad,69,Success
6,Jaipur,66,Success
7,Lucknow,131,Success
8,Bhubaneswar,33,Success
9,Indore,254,Success


In [36]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if file.endswith(".geojson"):
            print(os.path.join(root, file))

/kaggle/working/geo_data/schools_raw/bhubaneswar_osm_schools.geojson
/kaggle/working/geo_data/schools_raw/osm_schools_all_cities.geojson
/kaggle/working/geo_data/schools_raw/kolkata_osm_schools.geojson
/kaggle/working/geo_data/schools_raw/mumbai_osm_schools.geojson
/kaggle/working/geo_data/schools_raw/ahmedabad_osm_schools.geojson
/kaggle/working/geo_data/schools_raw/hyderabad_osm_schools.geojson
/kaggle/working/geo_data/schools_raw/indore_osm_schools.geojson
/kaggle/working/geo_data/schools_raw/pbf_extracted/chennai_schools.geojson
/kaggle/working/geo_data/schools_raw/pbf_extracted/mumbai_schools.geojson
/kaggle/working/geo_data/schools_raw/pbf_extracted/lucknow_schools.geojson
/kaggle/working/geo_data/schools_raw/pbf_extracted/ahmedabad_schools.geojson
/kaggle/working/geo_data/schools_raw/pbf_extracted/pune_schools.geojson
/kaggle/working/geo_data/schools_raw/pbf_extracted/kolkata_schools.geojson
/kaggle/working/geo_data/schools_raw/pbf_extracted/hyderabad_schools.geojson
/kaggle/wor

In [37]:
import os
import shutil
import zipfile
import pandas as pd

source = "/kaggle/working/geo_data/schools_raw/pbf_extracted"
backup = "/kaggle/working/HiMCM_backup"

os.makedirs(backup, exist_ok=True)

cities = {
    "pune": 256,
    "mumbai": 548,
    "hyderabad": 466,
    "chennai": 573,
    "kolkata": 119,
    "ahmedabad": 69,
    "jaipur": 66,
    "lucknow": 131,
    "bhubaneswar": 33,
    "indore": 254
}

# Copy the ten original files
for city in cities:
    filename = f"{city}_schools.geojson"
    src = os.path.join(source, filename)
    dst = os.path.join(backup, filename)

    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing: {src}")

    shutil.copy2(src, dst)

# Save extraction summary
summary = pd.DataFrame([
    {"city": city.title(), "schools": count, "status": "Success"}
    for city, count in cities.items()
])

summary.to_csv(
    os.path.join(backup, "extraction_summary.csv"),
    index=False
)

# Create ZIP backup
zip_path = "/kaggle/working/HiMCM_school_data_backup.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for filename in os.listdir(backup):
        filepath = os.path.join(backup, filename)
        z.write(filepath, arcname=filename)

print("Backup created successfully.")
print("Files:", len(cities) + 1)
print("Total school features:", sum(cities.values()))
print("ZIP:", zip_path)
print(summary)

Backup created successfully.
Files: 11
Total school features: 2515
ZIP: /kaggle/working/HiMCM_school_data_backup.zip
          city  schools   status
0         Pune      256  Success
1       Mumbai      548  Success
2    Hyderabad      466  Success
3      Chennai      573  Success
4      Kolkata      119  Success
5    Ahmedabad       69  Success
6       Jaipur       66  Success
7      Lucknow      131  Success
8  Bhubaneswar       33  Success
9       Indore      254  Success


In [39]:
import os

print(os.path.exists("/kaggle/working/HiMCM_school_data_backup.zip"))
print(os.listdir("/kaggle/working"))

True
['geo_data', 'cache', '.virtual_documents', 'HiMCM_quality_outputs.zip', 'HiMCM_backup', 'HiMCM_school_data_backup.zip']


In [40]:
from IPython.display import display, HTML

display(HTML("""
<a href="files/HiMCM_school_data_backup.zip" download>
    Download HiMCM school data backup
</a>
"""))

In [41]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if file == "HiMCM_school_data_backup.zip":
            print("FOUND:", os.path.join(root, file))

FOUND: /kaggle/working/HiMCM_school_data_backup.zip


In [42]:
from IPython.display import display, FileLink

display(FileLink("/kaggle/working/HiMCM_school_data_backup.zip"))

/kaggle/working/HiMCM_school_data_backup.zip